In [1]:
import os
import requests
from bs4 import BeautifulSoup
import numpy as np
import mne
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from collections import Counter
import seaborn as sns


# Step 1: Download CHB-MIT
BASE_URL = "https://physionet.org/files/chbmit/1.0.0/"
SAVE_DIR = "chbmit_data"
os.makedirs(SAVE_DIR, exist_ok=True)
SFREQ = 128
WINDOW_SEC = 30
HOP_SEC = 5

def download_file(url, save_path):
    if os.path.exists(save_path):
        return
    r = requests.get(url, stream=True)
    if r.status_code == 200:
        with open(save_path, 'wb') as f:
            for chunk in r.iter_content(1024):
                f.write(chunk)

def get_file_links(folder_url):
    r = requests.get(folder_url)
    soup = BeautifulSoup(r.text, "html.parser")
    links = []
    for a in soup.find_all("a"):
        href = a.get("href")
        if href and (href.endswith(".edf") or href.endswith(".txt")):
            links.append(folder_url + href)
    return links

patients = [f"chb{i:02d}" for i in range(1, 25)]
for patient_id in patients:
    folder_url = f"{BASE_URL}{patient_id}/"
    save_folder = os.path.join(SAVE_DIR, patient_id)
    os.makedirs(save_folder, exist_ok=True)
    for file_url in get_file_links(folder_url):
        filename = file_url.split("/")[-1]
        save_path = os.path.join(save_folder, filename)
        download_file(file_url, save_path)
    print(f"✅ Downloaded {patient_id}")


# Parse seizure annotations
def parse_summary(summary_file):
    seizures = {}
    filename, start, end = None, None, None
    with open(summary_file, "r") as f:
        for line in f:
            if "File Name" in line:
                filename = line.split(":")[-1].strip()
            elif "Seizure Start Time" in line:
                start = int(line.split(":")[-1].replace("seconds", "").strip())
            elif "Seizure End Time" in line:
                end = int(line.split(":")[-1].replace("seconds", "").strip())
                seizures.setdefault(filename, []).append((start, end))
    return seizures

# Data augmentation
def augment_window(win):
    aug = []
    aug.append(win + 0.01 * np.random.randn(*win.shape).astype(np.float32))  # noise
    aug.append(win * (1 + 0.1 * np.random.randn(win.shape[0], 1).astype(np.float32)))  # scaling
    shift = np.random.randint(-50, 50)
    aug.append(np.roll(win, shift, axis=1))
    return aug


# Detect most common channel count
def detect_common_channels(patients, base_dir=SAVE_DIR):
    channel_counts = []
    for patient_id in patients:
        save_folder = os.path.join(base_dir, patient_id)
        if not os.path.isdir(save_folder):
            continue
        for fname in os.listdir(save_folder):
            if fname.endswith(".edf"):
                edf_file = os.path.join(save_folder, fname)
                try:
                    raw = mne.io.read_raw_edf(edf_file, preload=False, verbose="error")
                    channel_counts.append(len(raw.ch_names))
                except Exception:
                    continue
    if len(channel_counts) == 0:
        raise RuntimeError("No EDF files found to detect channel counts.")
    most_common = Counter(channel_counts).most_common(1)[0][0]
    print(f"📊 Detected most common channel count: {most_common}")
    return most_common

target_channels = detect_common_channels(patients)


# Preprocess windows
def get_windows(edf_file, seizure_intervals, window_size=WINDOW_SEC, sfreq=SFREQ, hop_size=HOP_SEC, target_channels=target_channels):
    raw = mne.io.read_raw_edf(edf_file, preload=True, verbose="error")
    raw.resample(sfreq)
    data = raw.get_data().astype(np.float32)

    n_channels = data.shape[0]
    if n_channels > target_channels:
        data = data[:target_channels, :]
    elif n_channels < target_channels:
        pad = np.zeros((target_channels - n_channels, data.shape[1]), dtype=np.float32)
        data = np.vstack([data, pad])

    total_samples = data.shape[1]
    win_length = window_size * sfreq
    hop = hop_size * sfreq
    X, y = [], []

    for start in range(0, total_samples - win_length + 1, hop):
        end = start + win_length
        label = 0
        for (sz_start, sz_end) in seizure_intervals:
            if (start/sfreq >= sz_start - 30) and (end/sfreq <= sz_end):
                label = 1
                break
        win = data[:, start:end]
        X.append(win)
        y.append(label)

        if label == 1:
            for aug in augment_window(win):
                X.append(aug.astype(np.float32))
                y.append(1)

    if len(X) == 0:
        return np.empty((0, target_channels, win_length), dtype=np.float32), np.empty((0,), dtype=np.int64)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)


# Build dataset across patients
X_list, y_list = [], []
for patient_id in patients:
    save_folder = os.path.join(SAVE_DIR, patient_id)
    summary_file = os.path.join(save_folder, f"{patient_id}-summary.txt")
    if not os.path.exists(summary_file):
        continue
    annotations = parse_summary(summary_file)
    for fname, intervals in annotations.items():
        edf_file = os.path.join(save_folder, fname)
        if os.path.exists(edf_file):
            Xi, yi = get_windows(edf_file, intervals)
            if Xi.size > 0:
                X_list.append(Xi)
                y_list.append(yi)

if len(X_list) == 0:
    raise RuntimeError("No preprocessed windows were created. Check EDF files and summary annotations.")

X = np.vstack(X_list)
y = np.hstack(y_list)
print("✅ Final Dataset:", X.shape, y.shape, "| Seizure windows:", int(sum(y)), "Non-seizure windows:", int(len(y)-sum(y)))

X = (X - X.mean(axis=(1,2), keepdims=True)) / (X.std(axis=(1,2), keepdims=True) + 1e-8)

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)


# Train/test split & dataloaders
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=32, shuffle=False)


# Model
class CNNLSTM(nn.Module):
    def __init__(self, channels, time_steps):
        super(CNNLSTM, self).__init__()
        self.conv1 = nn.Conv1d(channels, 32, kernel_size=3)
        self.pool1 = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3)
        self.pool2 = nn.MaxPool1d(2)
        self.lstm = nn.LSTM(64, 64, batch_first=True)
        self.fc1 = nn.Linear(64, 64)
        self.fc2 = nn.Linear(64, 1)

    def _forward_features(self, x):
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))
        return x

    def forward(self, x):
        x = self._forward_features(x)  
        x = x.permute(0, 2, 1)        
        _, (h, _) = self.lstm(x)
        x = torch.relu(self.fc1(h[-1]))
        x = torch.sigmoid(self.fc2(x))
        return x.squeeze()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNLSTM(channels=X.shape[1], time_steps=X.shape[2]).to(device)
print(model)


# Train
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_losses = []
EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device).float()
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    avg_loss = running_loss / len(train_loader.dataset)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {avg_loss:.4f}")


# Evaluate
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        outputs = model(xb)
        all_preds.append(outputs.cpu().numpy())
        all_labels.append(yb.numpy())

all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)
y_pred = (all_preds > 0.5).astype(int)

print("\nConfusion Matrix:\n", confusion_matrix(all_labels, y_pred))
print("\nClassification Report:\n", classification_report(all_labels, y_pred, digits=4))

cm = confusion_matrix(all_labels, y_pred)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn + 1e-8)
specificity = tn / (tn + fp + 1e-8)
print(f"Sensitivity: {sensitivity:.4f}, Specificity: {specificity:.4f}")


# Compute ROC / PR metrics
fpr, tpr, _ = roc_curve(all_labels, all_preds)
roc_auc = auc(fpr, tpr)
precision, recall, _ = precision_recall_curve(all_labels, all_preds)


# Training loss curve
plt.figure(figsize=(6,5))
plt.plot(range(1, len(train_losses)+1), train_losses, marker='o', color='b')
plt.xlabel("Epochs")
plt.ylabel("Training Loss")
plt.title("Training Loss Curve")
plt.grid(True)
plt.tight_layout()
plt.savefig("training_loss_curve.png")
plt.close()


# Confusion matrix
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Seizure', 'Seizure'],
            yticklabels=['Non-Seizure', 'Seizure'])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png")
plt.close()


# ROC curve
plt.figure(figsize=(5,5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], 'r--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("roc_curve.png")
plt.close()


# Precision-Recall curve
plt.figure(figsize=(5,5))
plt.plot(recall, precision, label="Precision-Recall")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("pr_curve.png")
plt.close()


torch.save(model.state_dict(), "cnn_lstm_seizure_state_dict.pth")
print("Saved model: cnn_lstm_seizure_state_dict.pth")

✅ Downloaded chb01
✅ Downloaded chb02
✅ Downloaded chb03
✅ Downloaded chb04
✅ Downloaded chb05
✅ Downloaded chb06
✅ Downloaded chb07
✅ Downloaded chb08
✅ Downloaded chb09
✅ Downloaded chb10
✅ Downloaded chb11
✅ Downloaded chb12
✅ Downloaded chb13
✅ Downloaded chb14
✅ Downloaded chb15
✅ Downloaded chb16
✅ Downloaded chb17
✅ Downloaded chb18
✅ Downloaded chb19
✅ Downloaded chb20
✅ Downloaded chb21
✅ Downloaded chb22
✅ Downloaded chb23
✅ Downloaded chb24
📊 Detected most common channel count: 23
✅ Final Dataset: (28905, 23, 3840) (28905,) | Seizure windows: 1756 Non-seizure windows: 27149
CNNLSTM(
  (conv1): Conv1d(23, 32, kernel_size=(3,), stride=(1,))
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv1d(32, 64, kernel_size=(3,), stride=(1,))
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (lstm): LSTM(64, 64, batch_first=True)
  (fc1): Linear(in_features=64, out_features=64, bias=True)
  (fc2): Lin